# How to use AutoTools to make a dynamic toolkit

AutoTools wraps any SDK client that exposes CRUD operations (get, update, create, delete) and turns it into an LLM-enabled toolkit without a hand-written tool per call.

There are two core aspects to AutoTools:
1. `AutoToolWrapper` -- wraps the SDK you pass in
2. `CrudControls` -- controls the access an agent has to Create, Read, Update and Delete verbs

This notebook targets LangChain 1.x. Install what it needs with:

```bash
pip install "langchain-autotools" "langchain>=1.0" "langchain-anthropic>=1.0" boto3
```

First, we'll create a client SDK. Let's use AWS's `boto3` library. Ensure you have your credentials available or set them here.

In [ ]:
import getpass
import os

os.environ["AWS_ACCESS_KEY_ID"] = getpass.getpass(prompt="AWS Access Key ID: ")
os.environ["AWS_SECRET_ACCESS_KEY"] = getpass.getpass(prompt="AWS Secret Access Key: ")
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"

Let's validate that your S3 client is able to make an API call. This `list_buckets` command should match how many buckets are in your account.

In [ ]:
import boto3

s3 = boto3.client("s3")
buckets = s3.list_buckets()
print(len(buckets["Buckets"]))

Now we can pass this client into `AutoToolWrapper` to create the toolkit. The callable functions within the SDK become tools, available from `toolkit.get_tools()`.

_NOTE_ If your SDK has many callable functions, your tool list could exceed your model's context length. Use `CrudControls` to limit the tools your agent has access to.

To demonstrate this, we're limiting the `read_list` parameter to a single function, `list_buckets`. You can match more functions with a glob (`list_*`) or a regex (`r"^list_\w+$"`), or by listing several names.

In [ ]:
from langchain_autotools import AutoToolWrapper, CrudControls

crud_controls = CrudControls(
    read_list=["list_buckets"],
)

toolkit = AutoToolWrapper(client=s3, crud_controls=crud_controls)

print([tool.name for tool in toolkit.get_tools()])

Now we'll set the Anthropic API key.

In [ ]:
os.environ["ANTHROPIC_API_KEY"] = getpass.getpass(prompt="Anthropic API Key: ")

With the key set we can build the agent. LangChain 1.x uses `create_agent`, which returns a graph you invoke with a list of messages -- the old `AgentExecutor` / `create_structured_chat_agent` pair (and the `hwchase17/structured-chat-agent` prompt it needed) is no longer required.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    "anthropic:claude-opus-5",
    toolkit.get_tools(),
    system_prompt="You are an assistant that manages AWS resources for the user.",
)

Now we can use this agent to perform functions for us using the configured AWS `boto3` client. Let's ask the agent how many buckets we have.

In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "How many S3 buckets do I have?"}]}
)
print(result["messages"][-1].content)

Now let's have our agent create a bucket. We'll need to modify the `CrudControls` to include the `create_bucket` function from the `boto3` SDK, and enable create actions by setting `create` to `True`.

_WARNING_ This will create a bucket in your account! (Buckets are free to create, so long as you don't store data in them.)

In [ ]:
crud_controls = CrudControls(
    read_list=["list_buckets"],
    create=True,
    create_list=["create_bucket"],
)

toolkit = AutoToolWrapper(client=s3, crud_controls=crud_controls)

print([tool.name for tool in toolkit.get_tools()])

More complicated tools often require more capable models. We'll keep using Claude Opus 5 here.

In [ ]:
agent = create_agent(
    "anthropic:claude-opus-5",
    toolkit.get_tools(),
    system_prompt="You are an assistant that manages AWS resources for the user.",
)

In [ ]:
import random

rand = str(random.randint(0, 10000000))
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    f"Create a bucket named 'my-test-bucket-{rand}' in my account. "
                    "Be sure to pass the parameters as needed by the SDK."
                ),
            }
        ]
    }
)
print(result["messages"][-1].content)

To clean up the bucket you just created, run the block below. You could also clean up with the agent by modifying the `CrudControls` accordingly.

In [ ]:
s3.delete_bucket(Bucket=f"my-test-bucket-{rand}")